# Session 16: Build a Support Ticket Agent

Reuse the adapter exported by the fine-tuning practice to classify tickets. Observe the model’s decisions and the results returned by the local guide lookup.

## 1. Upload the adapter ZIP

In Colab, choose a **T4 GPU**. Open the **Files** pane and upload `session16-ticket-adapter.zip` from the fine-tuning notebook. In local Jupyter, put it in the notebook’s working directory. The ZIP contains the adapter weights, its PEFT configuration and the training method.

In [ ]:
import os
import subprocess
import sys
from zipfile import ZipFile

import torch

ADAPTER_ZIP = "session16-ticket-adapter.zip"
if not os.path.isfile(ADAPTER_ZIP):
    raise FileNotFoundError(f"Upload {ADAPTER_ZIP} from the fine-tuning practice first.")

# Restore the training method so QLoRA uses the same weight format.
with ZipFile(ADAPTER_ZIP) as archive:
    method = archive.read("training_method.txt").decode("utf-8").strip()

if method not in {"lora", "qlora"}:
    raise ValueError("Use an adapter ZIP exported by the fine-tuning notebook.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if method == "qlora" and device.type != "cuda":
    raise RuntimeError("Select a Colab T4 GPU for this QLoRA adapter.")

packages = ["transformers==4.57.1", "accelerate==1.10.1", "peft==0.17.1"]
if method == "qlora":
    packages.append("bitsandbytes==0.48.1")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print(f"Adapter: {method} | Device: {device}")

## 2. Load Qwen and the adapter

Load the original instruct checkpoint once. PEFT attaches the trained adapter. QLoRA uses the same 4-bit loading settings as the fine-tuning practice.

In [ ]:
from tempfile import TemporaryDirectory

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CHECKPOINT = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
dtype = torch.float16 if device.type == "cuda" else torch.float32
load_options = {"dtype": dtype}

if method == "qlora":
    load_options["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )
    load_options["device_map"] = {"": torch.cuda.current_device()}

base_model = AutoModelForCausalLM.from_pretrained(CHECKPOINT, **load_options)
if method == "lora":
    base_model.to(device)

with TemporaryDirectory() as adapter_dir, ZipFile(ADAPTER_ZIP) as archive:
    for filename in ("adapter_config.json", "adapter_model.safetensors"):
        archive.extract(filename, adapter_dir)
    # Attach the saved adapter to the checkpoint already in memory.
    model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

## 3. Load the support guides

The agent reads one local JSON file. Colab downloads it on the first run. To use the bundled guides from the local practice folder, set `GUIDE_JSON` to `../sample_data/support_guides.json`.

In [ ]:
import json
from urllib.request import urlretrieve

GUIDE_JSON = "support_guides.json"
# Download once so edits to the local guides are kept when this cell is rerun.
if not os.path.isfile(GUIDE_JSON):
    urlretrieve(
        "https://raw.githubusercontent.com/wlos/AI_SS/main/16-LLM-Fine-Tuning-and-Agents/sample_data/support_guides.json",
        GUIDE_JSON,
    )

with open(GUIDE_JSON, encoding="utf-8") as guide_file:
    support_guides = json.load(guide_file)

CATEGORIES = ("Account access", "Hardware", "Network", "Software")

## 4. Define the two model roles

Use the adapter for classification and temporarily disable it for routing and replies. The classification tool uses the same prompt as the fine-tuning practice and checks that the output contains an allowed category.

In [ ]:
from contextlib import nullcontext

from transformers import GenerationConfig


generation_settings = GenerationConfig(
    do_sample=False,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)


@torch.inference_mode()
def generate_reply(prompt, use_adapter=False, max_new_tokens=64):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    # Enable the adapter for classification and disable it for routing and replies.
    with nullcontext() if use_adapter else model.disable_adapter():
        tokens = model.generate(**inputs, generation_config=generation_settings, max_new_tokens=max_new_tokens)
    reply = tokens[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()


def classify_ticket(ticket):
    prompt = (
        f'Classify this support ticket as one of {", ".join(CATEGORIES)}. '
        'Return only a JSON object with the key "category".\n'
        f"Ticket: {ticket}"
    )
    reply = generate_reply(prompt, use_adapter=True, max_new_tokens=32)
    # Unwrap a Markdown code fence before validating the category.
    lines = reply.splitlines()
    if (
        len(lines) >= 3
        and lines[0].strip().casefold() in {"```", "```json"}
        and lines[-1].strip() == "```"
    ):
        reply = "\n".join(lines[1:-1])

    # Pass only an allowed category to the guide lookup.
    try:
        result = json.loads(reply)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and set(result) == {"category"}:
        category = result["category"]
        if category in CATEGORIES:
            return category
    return None

## 5. Run the agent

Qwen decides whether to reply directly or call the classifier. If the classifier returns an allowed category, Python looks up its guide and Qwen uses the result to reply. If classification fails, ask once for more details and retry with the updated ticket.

In [ ]:
import re


def run_ticket_agent(ticket, guides, *, is_follow_up=False):
    trace = {
        "decision": "CLASSIFY",
        "category": None,
        "guide": None,
        "needs_clarification": False,
    }

    # Route a new message. A follow-up to a clarification retries classification.
    if not is_follow_up:
        route = generate_reply(
            "Choose CLASSIFY for reported issues or requests for troubleshooting, even if vague. "
            "Choose DIRECT for greetings, thanks or general questions that need no troubleshooting. "
            'Examples: "Hello" → DIRECT. "My laptop will not charge" → CLASSIFY. '
            "Return only CLASSIFY or DIRECT.\n"
            f"Message: {ticket}",
            max_new_tokens=8,
        )
        # Only an explicit DIRECT choice bypasses the classifier.
        if re.match(r'^[\s`*"\']*DIRECT\b', route, flags=re.IGNORECASE):
            trace["decision"] = "DIRECT"
            trace["reply"] = generate_reply(
                "Reply briefly as a help desk assistant. "
                "You can help with account access, hardware, network and software issues.\n"
                f"Message: {ticket}"
            )
            return trace

    category = classify_ticket(ticket)
    trace["category"] = category
    if category is None:
        if is_follow_up:
            trace["reply"] = "I could not classify this ticket. Please contact the service desk."
        else:
            trace["decision"] = "ASK"
            trace["needs_clarification"] = True
            trace["reply"] = "Which service or device is affected, and what happened?"
        return trace

    # Use the validated category to select a guide for the reply.
    guide = guides.get(category)
    trace["guide"] = guide
    if guide is None:
        trace["reply"] = "I could not find a guide for this issue. Please contact the service desk."
        return trace

    trace["reply"] = generate_reply(
        f"Write a short support reply using only the guide below.\nTicket: {ticket}\nGuide: {guide}"
    )
    return trace

## 6. Chat with the ticket agent

Try **Hello, what can you help with?** for a direct reply. Then try **My student portal account is locked.** to use the classifier and guide. If classification fails and the agent asks for details, reply with the affected service or device and what happened. After a final reply, the next message starts a new ticket.

Type `/exit` to stop. Inspect `last_trace` to see the decision, category and guide.

In [ ]:
ticket_context = []
last_trace = None
print("Ticket support chat. Type /exit to stop.")

while True:
    message = input("You: ").strip()
    if message.casefold() == "/exit":
        break
    if not message:
        continue

    ticket_context.append(message)
    last_trace = run_ticket_agent(
        " ".join(ticket_context), support_guides,
        is_follow_up=len(ticket_context) > 1,
    )
    print(f"Agent: {last_trace['reply']}")

    # Keep earlier details while the agent is waiting for clarification.
    if not last_trace["needs_clarification"]:
        ticket_context.clear()